In [5]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base_path = '/content/drive/My Drive/MIMIC/'

df = pd.read_csv(base_path + 'training_dataset.csv')


from sklearn.model_selection import GroupKFold
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

import re  # Added regular expressions module
# 2. Separate Target and Group Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
cols_to_exclude = ['subject_id', 'hadm_id', 'icd_code', 'hospital_expire_flag']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

continuous_cols = ['anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
                   'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
                   'vital_mean_spo2', 'vital_mean_temp']


from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# 5. Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df['hospital_expire_flag'].value_counts()

,count
hospital_expire_flag,
0.0,17092
1.0,663


In [ ]:
y_train.value_counts()

,count
hospital_expire_flag,
0,13674
1,530


In [ ]:
y_test.value_counts()

,count
hospital_expire_flag,
0,3418
1,133


In [ ]:
# Model Training is down

In [ ]:
# XG BOOST

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Define hyperparameter distribution space
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1.0, 1.5, 2.0]  # Penalizes mistakes on imbalanced deceased cases
}

cv_strategy = GroupKFold(n_splits=5)
base_xgb = XGBClassifier(eval_metric='logloss', random_state=42)

print("Running fast RandomizedSearchCV over XGBoost parameters...")
xgb_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=xgb_params,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", xgb_search.best_params_)

# Evaluate on Holdout Test Set
best_xgb = xgb_search.best_estimator_
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

test_summary_xgb = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_xgb
})

patient_res_xgb = test_summary_xgb.groupby('hadm_id').mean()
preds_xgb = (patient_res_xgb['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized XGBoost Patient-Level Evaluation ---")
print(classification_report(patient_res_xgb['y_true'], preds_xgb))
print(f"XGBoost Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_xgb['y_true'], patient_res_xgb['y_prob']):.4f}")

Running fast RandomizedSearchCV over XGBoost parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'subsample': 0.7, 'scale_pos_weight': 1.5, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

--- Optimized XGBoost Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       686
         1.0       0.50      0.06      0.11        17

    accuracy                           0.98       703
   macro avg       0.74      0.53      0.55       703
weighted avg       0.97      0.98      0.97       703

XGBoost Patient-Level ROC-AUC Score: 0.9165


In [ ]:
# CAT BOOST

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# CatBoost parameters
cat_params = {
    'iterations': [150, 250, 350],
    'depth': [4, 5, 6, 7],
    'learning_rate': [0.02, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7]
}

cv_strategy = GroupKFold(n_splits=5)
# Using auto_class_weights='Balanced' organically scales minority cardiac death flags
base_cat = CatBoostClassifier(auto_class_weights='Balanced', verbose=0, random_seed=42)

print("Running fast RandomizedSearchCV over CatBoost parameters...")
cat_search = RandomizedSearchCV(
    estimator=base_cat,
    param_distributions=cat_params,
    n_iter=6,  # Slower to train, checking 6 combinations keeps loop fast
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

cat_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", cat_search.best_params_)

# Evaluate on Holdout Test Set
best_cat = cat_search.best_estimator_
y_prob_cat = best_cat.predict_proba(X_test)[:, 1]

test_summary_cat = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_cat
})

patient_res_cat = test_summary_cat.groupby('hadm_id').mean()
preds_cat = (patient_res_cat['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized CatBoost Patient-Level Evaluation ---")
print(classification_report(patient_res_cat['y_true'], preds_cat))
print(f"CatBoost Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_cat['y_true'], patient_res_cat['y_prob']):.4f}")

Running fast RandomizedSearchCV over CatBoost parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'learning_rate': 0.05, 'l2_leaf_reg': 1, 'iterations': 350, 'depth': 6}

--- Optimized CatBoost Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       686
         1.0       0.33      0.06      0.10        17

    accuracy                           0.97       703
   macro avg       0.66      0.53      0.54       703
weighted avg       0.96      0.97      0.97       703

CatBoost Patient-Level ROC-AUC Score: 0.9105


In [ ]:
# LIGHT GB

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# LightGBM parameter grids
lgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 50],
    'scale_pos_weight': [1.0, 1.5, 2.0]
}

cv_strategy = GroupKFold(n_splits=5)
base_lgb = lgb.LGBMClassifier(random_state=42, verbose=-1)

print("Running fast RandomizedSearchCV over LightGBM parameters...")
lgb_search = RandomizedSearchCV(
    estimator=base_lgb,
    param_distributions=lgb_params,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lgb_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", lgb_search.best_params_)

# Evaluate on Holdout Test Set
best_lgb = lgb_search.best_estimator_
y_prob_lgb = best_lgb.predict_proba(X_test)[:, 1]

test_summary_lgb = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_lgb
})

patient_res_lgb = test_summary_lgb.groupby('hadm_id').mean()
preds_lgb = (patient_res_lgb['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized LightGBM Patient-Level Evaluation ---")
print(classification_report(patient_res_lgb['y_true'], preds_lgb))
print(f"LightGBM Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_lgb['y_true'], patient_res_lgb['y_prob']):.4f}")

Running fast RandomizedSearchCV over LightGBM parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'scale_pos_weight': 1.0, 'num_leaves': 31, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}

--- Optimized LightGBM Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       686
         1.0       1.00      0.12      0.21        17

    accuracy                           0.98       703
   macro avg       0.99      0.56      0.60       703
weighted avg       0.98      0.98      0.97       703

LightGBM Patient-Level ROC-AUC Score: 0.9082


In [ ]:
# Random Forest Classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Random Forest parameters
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

cv_strategy = GroupKFold(n_splits=5)
# class_weight='balanced_subsample' matches bootstrap sample variations perfectly
base_rf = RandomForestClassifier(class_weight='balanced_subsample', random_state=42)

print("Running fast RandomizedSearchCV over Random Forest parameters...")
rf_search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=rf_params,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", rf_search.best_params_)

# Evaluate on Holdout Test Set
best_rf = rf_search.best_estimator_
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

test_summary_rf = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_rf
})

patient_res_rf = test_summary_rf.groupby('hadm_id').mean()
preds_rf = (patient_res_rf['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized Random Forest Patient-Level Evaluation ---")
print(classification_report(patient_res_rf['y_true'], preds_rf))
print(f"Random Forest Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_rf['y_true'], patient_res_rf['y_prob']):.4f}")

Running fast RandomizedSearchCV over Random Forest parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}

--- Optimized Random Forest Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       686
         1.0       0.00      0.00      0.00        17

    accuracy                           0.98       703
   macro avg       0.49      0.50      0.49       703
weighted avg       0.95      0.98      0.96       703

Random Forest Patient-Level ROC-AUC Score: 0.9018


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Multi-Layer Perceptron (Neural Network)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Multi-Layer Perceptron configuration grid
mlp_params = {
    'hidden_layer_sizes': [(32, 16), (64, 32), (50,)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

cv_strategy = GroupKFold(n_splits=5)
base_mlp = MLPClassifier(max_iter=250, activation='relu', solver='adam', early_stopping=True, random_state=42)

print("Running fast RandomizedSearchCV over MLP Neural Network parameters...")
mlp_search = RandomizedSearchCV(
    estimator=base_mlp,
    param_distributions=mlp_params,
    n_iter=5,  # Fewer iterations to keep backpropagation loops from locking the memory
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

mlp_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", mlp_search.best_params_)

# Evaluate on Holdout Test Set
best_mlp = mlp_search.best_estimator_
y_prob_mlp = best_mlp.predict_proba(X_test)[:, 1]

test_summary_mlp = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_mlp
})

patient_res_mlp = test_summary_mlp.groupby('hadm_id').mean()
preds_mlp = (patient_res_mlp['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized MLP Neural Network Patient-Level Evaluation ---")
print(classification_report(patient_res_mlp['y_true'], preds_mlp))
print(f"MLP Neural Network Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_mlp['y_true'], patient_res_mlp['y_prob']):.4f}")

Running fast RandomizedSearchCV over MLP Neural Network parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'learning_rate_init': 0.01, 'hidden_layer_sizes': (64, 32), 'alpha': 0.0001}

--- Optimized MLP Neural Network Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99       686
         1.0       0.29      0.12      0.17        17

    accuracy                           0.97       703
   macro avg       0.63      0.56      0.58       703
weighted avg       0.96      0.97      0.97       703

MLP Neural Network Patient-Level ROC-AUC Score: 0.8201


In [ ]:
# SVC  Linear Kernel

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# High-speed linear search grid
svm_linear_params = {
    'estimator__C': np.logspace(-3, 1, 6)
}

cv_strategy = GroupKFold(n_splits=5)

# LinearSVC is incredibly fast but natively lacks .predict_proba()
# We wrap it in CalibratedClassifierCV so you still get your Patient-Level ROC-AUC score smoothly
base_fast_linear = LinearSVC(
    loss='squared_hinge',
    class_weight='balanced',
    max_iter=1000,          # Hard cap prevents the solver from getting stuck in long loops
    random_state=42,
    dual=False              # Set dual=False when number of rows > number of columns for maximum speed
)
calibrated_base = CalibratedClassifierCV(base_fast_linear, cv=3)

print("Running ultra-fast RandomizedSearchCV over Linear SVM...")
svm_linear_search = RandomizedSearchCV(
    estimator=calibrated_base,
    param_distributions=svm_linear_params,
    n_iter=4,               # Targeted exploration to maintain high speed
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

svm_linear_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", svm_linear_search.best_params_)

# Evaluate on Holdout Test Set
best_svm_linear = svm_linear_search.best_estimator_
y_prob_svm_lin = best_svm_linear.predict_proba(X_test)[:, 1]

test_summary_svm_lin = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_svm_lin
})

patient_res_svm_lin = test_summary_svm_lin.groupby('hadm_id').mean()
preds_svm_lin = (patient_res_svm_lin['y_prob'] >= 0.5).astype(int)

print("\n--- High-Speed Linear SVM Patient-Level Evaluation ---")
print(classification_report(patient_res_svm_lin['y_true'], preds_svm_lin))
print(f"Linear SVM Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_svm_lin['y_true'], patient_res_svm_lin['y_prob']):.4f}")

Running ultra-fast RandomizedSearchCV over Linear SVM...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'estimator__C': np.float64(0.00630957344480193)}

--- High-Speed Linear SVM Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       686
         1.0       1.00      0.18      0.30        17

    accuracy                           0.98       703
   macro avg       0.99      0.59      0.64       703
weighted avg       0.98      0.98      0.97       703

Linear SVM Patient-Level ROC-AUC Score: 0.9286


In [ ]:
# SVM RBF Krnael

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

svm_rbf_params = {
    'C': np.logspace(-2, 1, 5),
    'gamma': ['scale', 'auto']
}

cv_strategy = GroupKFold(n_splits=5)
base_svm_rbf = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    max_iter=1000,          # Hard mathematical break protects your runtime environment
    random_state=42
)

print("Running high-speed RandomizedSearchCV over RBF SVM...")
svm_rbf_search = RandomizedSearchCV(
    estimator=base_svm_rbf,
    param_distributions=svm_rbf_params,
    n_iter=4,               # Caps search size to keep execution under a few minutes
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

svm_rbf_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", svm_rbf_search.best_params_)

# Evaluate on Holdout Test Set
best_svm_rbf = svm_rbf_search.best_estimator_
y_prob_svm_rbf = best_svm_rbf.predict_proba(X_test)[:, 1]

test_summary_svm_rbf = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_svm_rbf
})

patient_res_svm_rbf = test_summary_svm_rbf.groupby('hadm_id').mean()
preds_svm_rbf = (patient_res_svm_rbf['y_prob'] >= 0.5).astype(int)

print("\n--- High-Speed RBF SVM Patient-Level Evaluation ---")
print(classification_report(patient_res_svm_rbf['y_true'], preds_svm_rbf))
print(f"RBF SVM Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_svm_rbf['y_true'], patient_res_svm_rbf['y_prob']):.4f}")

Running high-speed RandomizedSearchCV over RBF SVM...


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'gamma': 'scale', 'C': np.float64(10.0)}

--- High-Speed RBF SVM Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99       686
         1.0       0.43      0.18      0.25        17

    accuracy                           0.97       703
   macro avg       0.70      0.59      0.62       703
weighted avg       0.97      0.97      0.97       703

RBF SVM Patient-Level ROC-AUC Score: 0.9026


In [ ]:
# Deep Learning Tabular Baseline (MLP Neural Network)

In [10]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Deep Tabular baseline parameters
mlp_params = {
    'hidden_layer_sizes': [(64, 32), (128, 64, 32), (100, 50)],
    'alpha': [0.0001, 0.005, 0.01],
    'learning_rate_init': [0.001, 0.005, 0.01]
}

cv_strategy = GroupKFold(n_splits=5)
base_mlp = MLPClassifier(max_iter=300, activation='relu', solver='adam', early_stopping=True, random_state=42)

print("Running fast RandomizedSearchCV over MLP Neural Network parameters...")
mlp_search = RandomizedSearchCV(
    estimator=base_mlp,
    param_distributions=mlp_params,
    n_iter=5,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

mlp_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", mlp_search.best_params_)

# Evaluate on Holdout Test Set
best_mlp = mlp_search.best_estimator_
y_prob_mlp = best_mlp.predict_proba(X_test)[:, 1]

test_summary_mlp = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_mlp
})

patient_res_mlp = test_summary_mlp.groupby('hadm_id').mean()
preds_mlp = (patient_res_mlp['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized MLP Neural Network Patient-Level Evaluation ---")
print(classification_report(patient_res_mlp['y_true'], preds_mlp))
print(f"MLP Neural Network Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_mlp['y_true'], patient_res_mlp['y_prob']):.4f}")

Running fast RandomizedSearchCV over MLP Neural Network parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'learning_rate_init': 0.01, 'hidden_layer_sizes': (100, 50), 'alpha': 0.0001}

--- Optimized MLP Neural Network Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99       686
         1.0       0.33      0.12      0.17        17

    accuracy                           0.97       703
   macro avg       0.66      0.56      0.58       703
weighted avg       0.96      0.97      0.97       703

MLP Neural Network Patient-Level ROC-AUC Score: 0.8083


In [ ]:
# Precision imporving of elasticnet  FAILED ONE

In [ ]:
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Expand 'C' search to favor stronger regularization (smaller values)
# Stronger regularization prevents weak predictors from creating false positives
param_distributions = {
    'C': np.logspace(-4, 1, 15),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0] # Included 1.0 (pure Lasso) to aggressively zero-out noise
}

cv_strategy = GroupKFold(n_splits=5)

# 2. CHANGE: Remove 'balanced' class weights.
# Letting the model see the raw imbalanced distributions forces it to be
# exceptionally selective and confident before predicting a Class 1.
base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight=None,  # Modifying this from 'balanced' to prioritize precision
    max_iter=300,       # SAGA solver needs a bit more room to converge with raw distributions
    random_state=42
)

print("Running fast RandomizedSearchCV optimized for PRECISION...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=12,          # Checked a few more combinations for tight convergence
    scoring='precision', # CHANGE: Natively optimizing for Precision instead of ROC-AUC
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

# Fit search on the training data
lr_search.fit(X_train, y_train)

Running fast RandomizedSearchCV optimized for PRECISION...


RandomizedSearchCV(cv=<generator object _BaseKFold.split at 0x7ddb73016350>,
                   estimator=LogisticRegression(max_iter=300,
                                                penalty='elasticnet',
                                                random_state=42,
                                                solver='saga'),
                   n_iter=12, n_jobs=-1,
                   param_distributions={'C': array([1.00000000e-04, 2.27584593e-04, 5.17947468e-04, 1.17876863e-03,
       2.68269580e-03, 6.10540230e-03, 1.38949549e-02, 3.16227766e-02,
       7.19685673e-02, 1.63789371e-01, 3.72759372e-01, 8.48342898e-01,
       1.93069773e+00, 4.39397056e+00, 1.00000000e+01]),
                                        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9,
                                                     1.0]},
                   random_state=42, scoring='precision')

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 6. Evaluate the Best Parameters on the Holdout Test Set
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

test_summary = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

# Collapse down to Patient-Level Metrics
patient_res = test_summary.groupby('hadm_id').mean()
preds = (patient_res['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized Patient-Level Evaluation ---")
print(classification_report(patient_res['y_true'], preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")

     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> C: 0.0316 | l1_ratio: 0.50

--- Optimized Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99       686
         1.0       1.00      0.12      0.21        17

    accuracy                           0.98       703
   macro avg       0.99      0.56      0.60       703
weighted avg       0.98      0.98      0.97       703

Optimized Patient-Level ROC-AUC Score: 0.9409


In [ ]:
# THE ABOVE ELASTICNET FAILED

In [ ]:
# ElasticNET

In [9]:
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
param_distributions = {
    'C': np.logspace(-2, 1, 10),              # Explores strong to moderate regularization
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]     # Balances feature dropping vs feature shrinking
}

# Use a single internal Group K-Fold split to evaluate parameters quickly
cv_strategy = GroupKFold(n_splits=5)

base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=200,
    random_state=42
)
print("Running fast RandomizedSearchCV over ElasticNet parameters...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,  # Only evaluates the top 8 random distinct combinations to save time
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)
# Fit search on the training data
lr_search.fit(X_train, y_train)


from sklearn.metrics import classification_report, roc_auc_score

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 6. Evaluate the Best Parameters on the Holdout Test Set
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

test_summary = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

# Collapse down to Patient-Level Metrics
patient_res = test_summary.groupby('hadm_id').mean()
preds = (patient_res['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized Patient-Level Evaluation ---")
print(classification_report(patient_res['y_true'], preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")

Running fast RandomizedSearchCV over ElasticNet parameters...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> C: 0.0464 | l1_ratio: 0.70

--- Optimized Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.99      0.91      0.95       686
         1.0       0.17      0.76      0.28        17

    accuracy                           0.90       703
   macro avg       0.58      0.84      0.61       703
weighted avg       0.97      0.90      0.93       703

Optimized Patient-Level ROC-AUC Score: 0.9234


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
# Lasso Logistic Regression ($L_1$ Regularization)   runner up

In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Lasso only tunes 'C' but uses it to drive weights to absolute zero
lasso_params = {
    'C': np.logspace(-3, 2, 12)
}

cv_strategy = GroupKFold(n_splits=5)
# Set penalty='l1' explicitly
base_lasso = LogisticRegression(
    penalty='l1',
    solver='saga',
    class_weight='balanced',
    max_iter=250,
    random_state=42
)

print("Running fast RandomizedSearchCV over Lasso (L1) Logistic Regression...")
lasso_search = RandomizedSearchCV(
    estimator=base_lasso,
    param_distributions=lasso_params,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", lasso_search.best_params_)

# Evaluate on Holdout Test Set
best_lasso = lasso_search.best_estimator_
y_prob_lasso = best_lasso.predict_proba(X_test)[:, 1]

test_summary_lasso = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_lasso
})

patient_res_lasso = test_summary_lasso.groupby('hadm_id').mean()
preds_lasso = (patient_res_lasso['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized Lasso (L1) Patient-Level Evaluation ---")
print(classification_report(patient_res_lasso['y_true'], preds_lasso))
print(f"Lasso Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_lasso['y_true'], patient_res_lasso['y_prob']):.4f}")

Running fast RandomizedSearchCV over Lasso (L1) Logistic Regression...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'C': np.float64(0.008111308307896872)}

--- Optimized Lasso (L1) Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       1.00      0.88      0.94       686
         1.0       0.16      0.88      0.27        17

    accuracy                           0.88       703
   macro avg       0.58      0.88      0.60       703
weighted avg       0.98      0.88      0.92       703

Lasso Patient-Level ROC-AUC Score: 0.9464


In [7]:
# Ridge Logistic Regression ($L_2$ Regularization)     Best one

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Ridge only tunes 'C' (Inverse regularization strength)
ridge_params = {
    'C': np.logspace(-3, 2, 12)
}

cv_strategy = GroupKFold(n_splits=5)
# Set penalty='l2' explicitly
base_ridge = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    class_weight='balanced',
    max_iter=250,
    random_state=42
)

print("Running fast RandomizedSearchCV over Ridge (L2) Logistic Regression...")
ridge_search = RandomizedSearchCV(
    estimator=base_ridge,
    param_distributions=ridge_params,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print("Best Parameters Found ->", ridge_search.best_params_)

# Evaluate on Holdout Test Set
best_ridge = ridge_search.best_estimator_
y_prob_ridge = best_ridge.predict_proba(X_test)[:, 1]

test_summary_ridge = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_ridge
})

patient_res_ridge = test_summary_ridge.groupby('hadm_id').mean()
preds_ridge = (patient_res_ridge['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized Ridge (L2) Patient-Level Evaluation ---")
print(classification_report(patient_res_ridge['y_true'], preds_ridge))
print(f"Ridge Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_ridge['y_true'], patient_res_ridge['y_prob']):.4f}")

Running fast RandomizedSearchCV over Ridge (L2) Logistic Regression...
     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> {'C': np.float64(0.008111308307896872)}

--- Optimized Ridge (L2) Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       1.00      0.91      0.95       686
         1.0       0.19      0.88      0.31        17

    accuracy                           0.90       703
   macro avg       0.59      0.89      0.63       703
weighted avg       0.98      0.90      0.93       703

Ridge Patient-Level ROC-AUC Score: 0.9441
